In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
# Load datasets
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

In [ ]:
# Explore credits dataset
print('Credits shape:', credits.shape)
print('Credits columns:', credits.columns.tolist())
credits.head(1)

In [ ]:
# Extract only the useful columns from movies
movies = movies[['id', 'title', 'genres', 'keywords', 'overview', 'tagline', 'vote_average']]
movies.head(1)

In [ ]:
# Check for missing values
print('Missing values in movies:')
print(movies.isnull().sum())

In [ ]:
# Merge the two datasets on movie id
movies = pd.merge(credits, movies, left_on='movie_id', right_on='id')
print('Merged shape:', movies.shape)
movies.head(1)

In [ ]:
# Drop the duplicate id column
movies = movies.drop('id', axis=1)
movies.head(2)

In [ ]:
# Parse string columns into Python lists using ast
for c in ['cast', 'crew', 'genres', 'keywords']:
    movies[c] = movies[c].apply(ast.literal_eval)

In [ ]:
# Extract only top 3 actors from cast
def get_top3_actors(cast):
    top3_actors = []
    for actor in cast:
        if actor['order'] in [0, 1, 2]:
            top3_actors.append(actor['name'])
    return top3_actors

movies['cast'] = movies['cast'].apply(get_top3_actors)
movies.head(2)

In [ ]:
# Extract only the director from crew
def get_director(crew):
    director = []
    for role in crew:
        if role['job'] == 'Director':
            director.append(role['name'])
            break
    return director

movies['crew'] = movies['crew'].apply(get_director)
movies = movies.rename(columns={'crew': 'director'})
movies.head(2)

In [ ]:
# Extract names from genres and keywords
def get_names(col):
    names = []
    for c in col:
        names.append(c['name'].lower())
    return names

movies['genres'] = movies['genres'].apply(get_names)
movies['keywords'] = movies['keywords'].apply(get_names)
movies.head(2)

In [ ]:
# Create tags by combining genres + keywords + cast + director
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['director']

# Remove spaces inside names so CountVectorizer treats them as one token
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(' ', '') for i in x])

# Convert list to a single string
movies['tags'] = movies['tags'].apply(lambda x: ' '.join(x))
movies['tags'][0]

In [ ]:
# Vectorize the tags using CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(movies['tags']).toarray()
print('Vectors shape:', vectors.shape)

In [ ]:
# Compute cosine similarity between all movies
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
print('Similarity matrix shape:', similarity.shape)
print('Sample similarity scores for movie 0:', similarity[0][:10])

In [ ]:
# Test the recommendation logic
def recommend(movie):
    index = movies[movies['title'].str.lower().str.strip() == movie].index[0]
    scores = similarity[index]
    movie_list = sorted(list(enumerate(scores)), reverse=True, key=lambda x: x[1])[1:11]
    titles = []
    for i in movie_list:
        titles.append(movies.iloc[i[0]].title)
    return titles

# Quick test
print(recommend('inception'))

In [ ]:
# Save the model and movie data as pickle files
import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('model.pkl', 'wb'))

print('movies.pkl and model.pkl saved successfully!')